# What is hyDe?

- HyDE(Hypothetical Document Embeddings) is a retrival technique where, instead of embedding the user's query directly, you first generate a hypothetical answer (document) to the query using LLM - and then embed that hypothetical document to search your vector store.

HyDE bridges the gap between user intent and relevant content, especially when:
1. Queries are short
2. Language mismatch between query and documents
3. you wantto retrive based on answer content, not question words

In [1]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
loader  = WikipediaLoader(query="Steve Jobs", load_max_docs=20)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(texts, embeddings)
retriever = db.as_retriever()


c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
C:\Users\mendh\AppData\Local\Temp\ipykernel_6616\1468990835.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip ins

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import os
os.environ["Groq_API_Key"] = os.getenv("Groq_API_Key")
from langchain.chat_models import init_chat_model
llm  = init_chat_model(model_provider="groq", model="llama-3.1-8b-instant")

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

def hyde(query):
    prompt = SystemMessagePromptTemplate.from_template("""Imagine you are an expert writing a detailed explanation on the following topic: {query} create a hypothetical answer to the topic""")
    chat_prompt = ChatPromptTemplate.from_messages([prompt])
    messages = chat_prompt.format_messages(query=query)
    print("Generated Hypothetical Answer Prompt:", messages)
    response = llm.invoke(messages)
    return response.content

In [5]:
query = "life of Steve Jobs"
hypothetical_answer = hyde(query)
print("Hypothetical Answer:", hypothetical_answer)

Generated Hypothetical Answer Prompt: [SystemMessage(content='Imagine you are an expert writing a detailed explanation on the following topic: life of Steve Jobs create a hypothetical answer to the topic', additional_kwargs={}, response_metadata={})]
Hypothetical Answer: **The Life of Steve Jobs: A Visionary Entrepreneur and Innovator**

Steve Jobs (1955-2011) was a pioneering entrepreneur, inventor, and designer who co-founded Apple Inc., one of the world's most valuable and influential technology companies. His life was marked by innovation, perseverance, and a passion for revolutionizing the way people interact with technology. Here's a comprehensive look at the life of Steve Jobs:

**Early Life (1955-1972)**

Steve Jobs was born on February 24, 1955, in San Francisco, California, to two University of Wisconsin graduate students, Joanne Schieble and Abdulfattah "John" Jandali. His biological parents gave him up for adoption, and he was adopted by Paul and Clara Jobs, a machinist and

In [6]:
retrived = retriever.invoke(hypothetical_answer)
print("Retrieved Documents:", retrived)

Retrieved Documents: [Document(metadata={'source': 'https://en.wikipedia.org/wiki/List_of_depictions_of_Steve_Jobs', 'summary': 'Steve Jobs was an American pioneer of the personal computer revolution of the 1970s who, along with Steve Wozniak and Ronald Wayne, founded Apple Computer. Before and after his death in 2011, Jobs was known as a counter-culture figure within the computer industry, and as a perfectionist who could be demanding of his colleagues and employees—sometimes to the point of cruelty.\nJobs\'s official biographer, Walter Isaacson, described him as a "creative entrepreneur whose passion for perfection and ferocious drive revolutionized six industries: personal computers, animated movies, music, phones, tablet computing, and digital publishing".', 'title': 'List of depictions of Steve Jobs'}, page_content='Steve Jobs was an American pioneer of the personal computer revolution of the 1970s who, along with Steve Wozniak and Ronald Wayne, founded Apple Computer. Before and 

### Langchain-HypotheticalDocumentEmbedder

In [7]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.document_loaders import TextLoader, DirectoryLoader

In [8]:
loader = DirectoryLoader(r"C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files", glob="**/*.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
hyde_embedder = HypotheticalDocumentEmbedder.from_llm(
    base_embeddings=embeddings,
      llm=llm,
      prompt_key ="web_search")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
vector = Chroma.from_documents(
    documents=texts,
    embedding=hyde_embedder,
    
)

In [10]:
rag_prompt = PromptTemplate.from_template(
    """Use the context below to answer the question.
    
    Context:{context}
    Question:{input}"""
)
rag_chain = create_stuff_documents_chain(llm=llm,prompt=rag_prompt)

In [11]:
def hyde_rag_pipeline(query):
    docs = vector.similarity_search(query,k=4)
    print(docs)
    response= rag_chain.invoke({"input":query,
                                "context":docs})
    return response

In [12]:
query = "what is Machine learning"
answer = hyde_rag_pipeline(query)
print(answer)

[Document(metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Tom M. Mitchell provided a widely quoted, more formal definition of the algorithms studied in the machine learning field: "A computer program is said to learn from experience E with respect to some class of tasks T and performance measure P if its performance at tasks in T, as measured by P, improves with experience E."[15] This definition of the tasks in which machine learning is concerned offers a fundamentally operational definition rather than defining the field in cognitive terms. This follows Alan Turing\'s proposal in his paper "Computing Machinery and Intelligence", in which the question, "Can machines think?", is replaced with the question, "Can machines do what we (as thinking entities) can do?".[16]\n\nModern-day Machine Learning algorithms are broken into 3 algorithm types: Supervised Learning Algorithms, Unsupervised Learning